# Diabetes Healthcare Analysis & Readmission Prediction

## 1. Project Objective

## 5. Exploratory Analysis

## 6. Feature Engineering

## 7. Predictive Modelling

## 8. Model Evaluation

## 9. Model Interpretation

## 10. Limitations

## 11. Conclusion

## Dataset Overview

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/diabetic_data.csv")

df.shape

(101766, 50)

In [2]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

## Data Inspection

In [4]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())

Rows: 101766
Columns: 50
Duplicate rows: 0


In [5]:
df.isna().sum().sort_values(ascending=False).head(15)

max_glu_serum    96420
A1Cresult        84748
encounter_id         0
nateglinide          0
glimepiride          0
acetohexamide        0
glipizide            0
glyburide            0
tolbutamide          0
pioglitazone         0
rosiglitazone        0
acarbose             0
miglitol             0
troglitazone         0
tolazamide           0
dtype: int64

In [6]:
df["readmitted"].value_counts(dropna=False)

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [7]:
missing_marker_counts = (df == "?").sum().sort_values(ascending=False)

missing_marker_counts.head(20)

weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
encounter_id             0
tolazamide               0
glyburide                0
tolbutamide              0
pioglitazone             0
rosiglitazone            0
acarbose                 0
miglitol                 0
troglitazone             0
citoglipton              0
examide                  0
acetohexamide            0
insulin                  0
dtype: int64

In [8]:
missing_marker_percent = (
    (df == "?").mean() * 100
).sort_values(ascending=False)

missing_marker_percent.head(20)

weight               96.858479
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
encounter_id          0.000000
tolazamide            0.000000
glyburide             0.000000
tolbutamide           0.000000
pioglitazone          0.000000
rosiglitazone         0.000000
acarbose              0.000000
miglitol              0.000000
troglitazone          0.000000
citoglipton           0.000000
examide               0.000000
acetohexamide         0.000000
insulin               0.000000
dtype: float64

In [9]:
for column in ["weight", "payer_code", "medical_specialty", "race", "gender"]:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False).head(10))


weight
weight
?            98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

payer_code
payer_code
?     40256
MC    32439
HM     6274
SP     5007
BC     4655
MD     3532
CP     2533
UN     2448
CM     1937
OG     1033
Name: count, dtype: int64

medical_specialty
medical_specialty
?                             49949
InternalMedicine              14635
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Name: count, dtype: int64

race
race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dty

In [10]:
id_columns = ["encounter_id", "patient_nbr"]

for column in id_columns:
    print(f"\n{column}")
    print("Unique values:", df[column].nunique())
    print("Duplicate values:", df[column].duplicated().sum())


encounter_id
Unique values: 101766
Duplicate values: 0

patient_nbr
Unique values: 71518
Duplicate values: 30248


In [11]:
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False).head(10))


--- race ---
race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

--- gender ---
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

--- age ---
age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64

--- weight ---
weight
?            98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

--- payer_code ---
payer_code
?     40256
MC    32439
HM     6274
SP     5007
BC     4655
MD     3532
CP     2533
UN     2448
CM     1937
OG     1033
Name: count, dtype: int64

--- medical_specialty ---
medical_special

## Data Cleaning

In [12]:
import numpy as np

df_clean = df.copy()

# Preserve patient identifier for later patient-level splitting.
# It will not be used as a model feature.

# Convert dataset-specific missing markers to NaN
missing_marker_columns = [
    "race",
    "weight",
    "payer_code",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3"
]

df_clean[missing_marker_columns] = (
    df_clean[missing_marker_columns]
    .replace("?", np.nan)
)

df_clean.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [13]:
# Remove variables with extremely high missingness
df_clean = df_clean.drop(columns=["weight"])

# Remove constant variables
constant_columns = [
    column
    for column in df_clean.columns
    if df_clean[column].nunique(dropna=False) <= 1
]

constant_columns

['examide', 'citoglipton']

In [14]:
df_clean = df_clean.drop(columns=constant_columns)

print("Removed constant columns:", constant_columns)
print("New shape:", df_clean.shape)

Removed constant columns: ['examide', 'citoglipton']
New shape: (101766, 47)


In [15]:
categorical_missing_columns = [
    "race",
    "payer_code",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3"
]

df_clean[categorical_missing_columns] = (
    df_clean[categorical_missing_columns]
    .fillna("Unknown")
)

In [16]:
df_clean[categorical_missing_columns].isna().sum()

race                 0
payer_code           0
medical_specialty    0
diag_1               0
diag_2               0
diag_3               0
dtype: int64

In [17]:
age_mapping = {
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95
}

df_clean["age_numeric"] = df_clean["age"].map(age_mapping)

In [18]:
df_clean[["age", "age_numeric"]].drop_duplicates().sort_values("age_numeric")

,age,age_numeric
0,[0-10),5
1,[10-20),15
2,[20-30),25
3,[30-40),35
4,[40-50),45
5,[50-60),55
6,[60-70),65
7,[70-80),75
8,[80-90),85
9,[90-100),95


In [19]:
df_clean["readmission_30d"] = (
    df_clean["readmitted"] == "<30"
).astype(int)

df_clean["readmission_30d"].value_counts()

readmission_30d
0    90409
1    11357
Name: count, dtype: int64

In [20]:
pd.crosstab(
    df_clean["readmitted"],
    df_clean["readmission_30d"]
)

readmission_30d,0,1
readmitted,,
<30,0,11357
>30,35545,0
NO,54864,0


In [21]:
for column in ["diag_1", "diag_2", "diag_3"]:
    print(
        column,
        "unique codes:",
        df_clean[column].nunique()
    )

diag_1 unique codes: 717
diag_2 unique codes: 749
diag_3 unique codes: 790


In [22]:
df_clean[["diag_1", "diag_2", "diag_3"]].head(10)

,diag_1,diag_2,diag_3
0,250.83,Unknown,Unknown
1,276,250.01,255
2,648,250,V27
3,8,250.43,403
4,197,157,250
5,414,411,250
6,414,411,V45
7,428,492,250
8,398,427,38
9,434,198,486


In [23]:
def get_icd9_category(code):
    if pd.isna(code) or code == "Unknown":
        return "Unknown"
    
    code = str(code)
    
    if code.startswith("V"):
        return "V-code"
    elif code.startswith("E"):
        return "E-code"
    
    try:
        numeric_code = float(code)
        
        if 390 <= numeric_code < 460:
            return "Circulatory"
        elif 460 <= numeric_code < 520:
            return "Respiratory"
        elif 520 <= numeric_code < 580:
            return "Digestive"
        elif 580 <= numeric_code < 630:
            return "Genitourinary"
        elif 630 <= numeric_code < 680:
            return "Pregnancy"
        elif 680 <= numeric_code < 710:
            return "Skin"
        elif 710 <= numeric_code < 740:
            return "Musculoskeletal"
        elif 740 <= numeric_code < 760:
            return "Congenital"
        elif 760 <= numeric_code < 780:
            return "Perinatal"
        elif 780 <= numeric_code < 800:
            return "Symptoms"
        elif 800 <= numeric_code < 1000:
            return "Injury"
        elif 250 <= numeric_code < 251:
            return "Diabetes"
        else:
            return "Other"
            
    except ValueError:
        return "Other"

In [24]:
for column in ["diag_1", "diag_2", "diag_3"]:
    df_clean[f"{column}_category"] = df_clean[column].apply(get_icd9_category)

In [25]:
for column in ["diag_1_category", "diag_2_category", "diag_3_category"]:
    print(f"\n--- {column} ---")
    print(df_clean[column].value_counts())


--- diag_1_category ---
diag_1_category
Circulatory        30336
Other              13479
Respiratory        10407
Digestive           9208
Diabetes            8757
Symptoms            7636
Injury              6974
Genitourinary       5078
Musculoskeletal     4957
Skin                2530
V-code              1644
Pregnancy            687
Congenital            51
Unknown               21
E-code                 1
Name: count, dtype: int64

--- diag_2_category ---
diag_2_category
Circulatory        31365
Other              19570
Diabetes           12794
Respiratory        10251
Genitourinary       7987
Symptoms            4632
Digestive           3962
Skin                3596
Injury              2428
V-code              1805
Musculoskeletal     1764
E-code               731
Pregnancy            415
Unknown              358
Congenital           108
Name: count, dtype: int64

--- diag_3_category ---
diag_3_category
Circulatory        29918
Other              20260
Diabetes           17157


In [26]:
df_clean.shape

(101766, 52)

In [27]:
df_clean.isna().sum().sort_values(ascending=False).head(15)

max_glu_serum          96420
A1Cresult              84748
encounter_id               0
insulin                    0
glipizide                  0
glyburide                  0
tolbutamide                0
pioglitazone               0
rosiglitazone              0
acarbose                   0
miglitol                   0
troglitazone               0
tolazamide                 0
glyburide-metformin        0
glimepiride                0
dtype: int64

In [28]:
df_clean.to_csv(
    "../data/cleaned/diabetes_cleaned_intermediate.csv",
    index=False
)

print("Saved:", df_clean.shape)

Saved: (101766, 52)
